# B2-024 — Practice p06

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `aio-ml:F1-scientific-python`, `aio-ml:F4-matrices`, `aio-ml:F3-multivar-calculus`, `aio-ml:F5-probability`, `aio-ml:C1-ml-fundamentals`, `aio-ml:C2-linear-models`, `aio-ml:C3-gradient-descent`, `aio-ml:C6-neural-networks`, `aio-ml:C7-pytorch`, `aio-ml:C10-cnn-transfer`, `aio-ml:C11-competition-craft`, `aio-ml:C8-neural-training`, `aio-ml:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [aio-ml:F1-scientific-python](../../../../aio-ml/units/01-F1-scientific-python/lesson.ipynb), [aio-ml:F4-matrices](../../../../aio-ml/units/04-F4-matrices/lesson.ipynb), [aio-ml:F3-multivar-calculus](../../../../aio-ml/units/03-F3-multivar-calculus/lesson.ipynb), [aio-ml:F5-probability](../../../../aio-ml/units/05-F5-probability/lesson.ipynb), [aio-ml:C1-ml-fundamentals](../../../../aio-ml/units/06-C1-ml-fundamentals/lesson.ipynb), [aio-ml:C2-linear-models](../../../../aio-ml/units/07-C2-linear-models/lesson.ipynb), [aio-ml:C3-gradient-descent](../../../../aio-ml/units/08-C3-gradient-descent/lesson.ipynb), [aio-ml:C6-neural-networks](../../../../aio-ml/units/11-C6-neural-networks/lesson.ipynb), [aio-ml:C7-pytorch](../../../../aio-ml/units/13-C7-pytorch/lesson.ipynb), [aio-ml:C10-cnn-transfer](../../../../aio-ml/units/17-C10-cnn-transfer/lesson.ipynb), [aio-ml:C11-competition-craft](../../../../aio-ml/units/18-C11-competition-craft/lesson.ipynb), [aio-ml:C8-neural-training](../../../../aio-ml/units/15-C8-neural-training/lesson.ipynb), [aio-ml:C12-classical-models](../../../../aio-ml/units/19-C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** intro · **Time budget:** 50 minutes  
**Concepts:** gpu-colab-l4-workflow

## Task

Implement three device-agnostic helpers and verify them on a CPU.

**Contract.**

* `get_device()` returns `torch.device("cuda")` when `torch.cuda.is_available()` and `torch.device("cpu")` otherwise.
* `move_batch(batch, device)` returns a new structure of the same type in which every tensor is replaced by `tensor.to(device)`: tensors are moved; `tuple` and `list` are rebuilt with the same type and order; `dict` is rebuilt with the same keys; any other object (strings, ints, `None`) is returned unchanged. Nested structures are handled recursively. The input structure must not be modified in place.
* `train_step(model, batch, optimizer, device, amp_dtype)` takes `batch = (images, labels)`, moves it with `move_batch`, runs `optimizer.zero_grad(set_to_none=True)`, computes `logits = model(images)` and `loss = F.cross_entropy(logits, labels)` (both inside `torch.autocast(device_type=device.type, dtype=amp_dtype)` when `amp_dtype` is not `None`, and with no autocast at all when it is `None`), then runs `loss.backward()` and `optimizer.step()` outside the autocast context. It returns `{"loss": float, "logits_dtype": torch.dtype}`, where `"loss"` is the detached loss value. The model is assumed to be on `device` already.

**Data and model for the probes.** Load the unit loader with the supplied cell. Use `images, labels = capstone_data.train_rows("shapes_supervised")` and the fixed batch `(images[:64], labels[:64])`. Build the model with `torch.manual_seed(SEED)` immediately followed by `model = capstone_data.ShapeCNN()`, and `optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`.

**Probes (CPU).**

1. `get_device().type` is `"cuda"` exactly when `torch.cuda.is_available()`.
2. `move_batch(NESTED, torch.device("cpu"))` returns a `dict` with the same keys; the value types are `torch.Tensor`, `list`, `tuple`, `str`; the strings and the integer inside the list are unchanged; every tensor equals the original (`torch.equal`).
3. `move_batch(NESTED, torch.device("meta"))` places **every** tensor on the `meta` device (a data-free device available on every machine), and `NESTED`'s own tensors are still on the CPU afterwards.
4. With `amp_dtype=None`, five calls of `train_step` on the fixed batch return `logits_dtype == torch.float32`, finite losses, and a fifth loss strictly smaller than the first.
5. With a fresh model and optimizer (same seed) and `amp_dtype=torch.bfloat16`, one call returns `logits_dtype == torch.bfloat16`, every model parameter is still `torch.float32`, and every parameter's `.grad` is `torch.float32`. (CPU `bfloat16` is used here for correctness only, not speed.)
6. One config dict drives one code path: write `run(config)` that builds the model and optimizer as above on `torch.device(config["device"])` and performs `config["steps"]` calls of `train_step` with `config["amp_dtype"]` on the batch `(images[:config["batch_size"]], labels[:config["batch_size"]])`, where `images, labels` are the training rows above when `config["simulated_rows"] == 0` and `capstone_data.simulate("shapes_supervised", config["simulated_rows"], seed=1)` otherwise; it returns the list of losses.
   Every device, precision, and size knob is read from `config`, so the accelerator extension changes only the dictionary. `run(CPU_CONFIG)` with the supplied `CPU_CONFIG` must return 5 finite losses.

**Required answer-check assertions.** Probes 1–6, including the `meta` placement check for every tensor in probe 3 and the dtype checks of probe 5.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
# Boilerplate: find and load this unit's data module
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("aio-dl/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

NESTED = {
    "images": torch.zeros(2, 1, 8, 8),
    "extras": [torch.arange(3), "shape-batch", 7],  # torch.arange(3): 0, 1, 2
    "pair": (torch.ones(2), torch.tensor([1, 2])),
    "name": "probe",
}
CPU_CONFIG = {"device": "cpu", "amp_dtype": None, "steps": 5, "batch_size": 64, "simulated_rows": 0}

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

Run the same `run(config)` with a replacement config dict on a Colab L4 runtime. `L4_CONFIG` has exactly the keys of `CPU_CONFIG`; only the values change, and `train_step`, `move_batch`, and `run` are the code you tested on the CPU (the 2048-row batch comes from `simulated_rows`).

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "steps": 200, "batch_size": 2048, "simulated_rows": 2048}
losses = run(L4_CONFIG)
```

**What to observe.** `get_device()` reports `cuda`; `logits_dtype` is `torch.bfloat16`; time per step for `run({**L4_CONFIG, "amp_dtype": None})` versus `run(L4_CONFIG)` at batch 2048 (on the L4, `bfloat16` should be faster, unlike on the CPU); and `torch.cuda.max_memory_allocated()` after one step for both settings.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.